In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("../data/raw/lebron_game_logs.csv")

In [ ]:
df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"])
df = df.sort_values("GAME_DATE").reset_index(drop=True)

In [ ]:
stats = ["PTS", "REB", "AST", "MIN"]

for stat in stats:
    # previous game
    df[f"{stat}_LAG_1"] = df[stat].shift(1)
    
    # avgs of the 5 and 10 previous games
    df[f"{stat}_AVG_5"] = (
        df[stat].shift(1).rolling(window=5).mean()
    )
    
    df[f"{stat}_AVG_10"] = (
        df[stat].shift(1).rolling(window=10).mean()
    )

# rest days
df["REST_DAYS"] = (
    df.groupby("SEASON_ID")["GAME_DATE"]
    .diff()
    .dt.days - 1
)

# home vs away
df["HOME"] = (
    df["MATCHUP"].str.contains("vs.", regex=False)
).astype(int)

df.head(15)

In [ ]:
# FEATURES
feature_cols = [
    "PTS_LAG_1", "PTS_AVG_5", "PTS_AVG_10",
    "REB_LAG_1", "REB_AVG_5", "REB_AVG_10",
    "AST_LAG_1", "AST_AVG_5", "AST_AVG_10",
    "MIN_LAG_1", "MIN_AVG_5", "MIN_AVG_10",
    "REST_DAYS", "HOME"
]

# TARGETS
target_cols = ["PTS", "REB", "AST", "MIN"]

# REMOVING LINES WITH MISSING DATA
df_model = df.dropna(
    subset=feature_cols + target_cols
).copy()

# CONVERTS TO INT
df_model["REST_DAYS"] = df_model["REST_DAYS"].astype(int)

print(f"Original dataframe: {len(df)} lines")
print(f"Dataframe post cleaning: {len(df_model)} lines")
print(f"Remaining nulls: {df_model[feature_cols + target_cols].isna().sum().sum()}")